In [0]:
from pyspark.sql.functions import *

In [0]:
import sys
import importlib
sys.path.append('/Workspace/Users/kiranmanam9490@outlook.com/customer360')

import common.configuration
importlib.reload(common.configuration)
common.configuration.dbutils = dbutils

from common.configuration import Configuration

conf = Configuration()

In [0]:
cust_df = spark.read.format('delta').load(conf.base_url + '/silver/customers')
ord_df = spark.read.format('delta').load(conf.base_url + '/silver/orders')
pay_df = spark.read.format('delta').load(conf.base_url + '/silver/payments')
tickets_df = spark.read.format('delta').load(conf.base_url + '/silver/support_tickets')
web_df = spark.read.format('delta').load(conf.base_url + '/silver/web_activities')

In [0]:
customer360_df = cust_df.alias('c').join(ord_df.alias('o'), 'customer_id','left')\
    .join(pay_df.alias('p'), 'customer_id','left')\
    .join(tickets_df.alias('t'), 'customer_id','left')\
    .join(web_df.alias('w'), 'customer_id')\
    .select('c.customer_id','c.name','c.email','c.gender','c.dob','c.location','o.order_id','o.order_date',col('o.amount').alias('order_amount'),col('status').alias('order_status'),'payment_id','payment_date','payment_method','payment_status',col('p.amount').alias('payment_amount'),'ticket_id','issue_type','ticket_date','resolution_status','session_id','page_viewed','session_time','device_type')



customer360_df.write.format('delta').mode('overwrite').save(conf.base_url + '/gold/customer360')

In [0]:

customer360 = spark.read.format('delta').load(conf.base_url + '/gold/customer360')

In [0]:
display(customer360)